<a href="https://colab.research.google.com/github/JASotomayor/Pistacho-Climatic-Comparation/blob/main/Copia_de_MX_Vs_AS_II_1_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BLOQUE 1 — Configuración Global y Ubicaciones

In [32]:
# --- Crear config.py con constantes globales ---
config_code = """
# config.py

# Período de análisis
START_YEAR = 2000
END_YEAR = 2024

# Umbrales climáticos
CHILL_THRESHOLD = 7.0           # Horas frío < 7°C
HEAT_STRESS_THRESHOLD = 35.0    # Estrés térmico ≥ 35°C
FROST_THRESHOLD = 0.0           # Evento de helada ≤ 0°C
GDD_BASE = 7.0                  # Temperatura base para GDD (Growing Degree Days)

# Constantes FAO-56 para ETo
ALBEDO = 0.23                   # Reflectancia del suelo
SIGMA = 4.903e-9                # Constante de Stefan-Boltzmann (MJ·K⁻⁴·m⁻²·día⁻¹)
GSC = 0.0820                    # Constante solar (MJ·m⁻²·min⁻¹)
Z = 10                          # Altura del viento (m)

# Tabla de Weinberger para frío estimado
WEIM_T = [13.2, 12.3, 11.4, 10.6, 9.8, 8.3, 7.6, 6.9, 6.3]
WEIM_HF = [450, 550, 650, 750, 850, 950, 1050, 1150, 1350]
"""

with open("config.py", "w") as f:
    f.write(config_code)

print("✅ Archivo config.py creado correctamente.")

# --- Crear locations.py con coordenadas y hemisferio ---
locations_code = """
# locations.py

LOCATIONS = {
    "Chihuahua_MX": {
        "lat": 27.0398,
        "lon": -105.2063,
        "hemisphere": "north"
    },
    "Saudi Arabia Point II.Sakaka": {
        "lat": 29.7973,
        "lon": 39.9536,
        "hemisphere": "north"
    },
    # Puedes añadir más localizaciones aquí si lo necesitas
}

# Lista de sitios seleccionados para análisis (filtrables)
SELECTED_SITES = list(LOCATIONS.keys())
"""

with open("locations.py", "w") as f:
    f.write(locations_code)

print("✅ Archivo locations.py creado correctamente.")

✅ Archivo config.py creado correctamente.
✅ Archivo locations.py creado correctamente.


# BLOQUE 2 — Crear archivo data_fetcher.py

In [2]:
data_fetcher_code = """
# data_fetcher.py

import requests
import time

def fetch_hourly_era5(lat, lon, start_date, end_date, variables=None, retries=3):
    \"""
    Descarga datos horarios de ERA5 vía Open-Meteo.
    Devuelve un diccionario con arrays por variable.
    \"""
    if variables is None:
        variables = [
            "temperature_2m", "relativehumidity_2m",
            "windspeed_10m", "shortwave_radiation", "precipitation"
        ]

    var_str = ",".join(variables)
    url = (
        "https://archive-api.open-meteo.com/v1/era5?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start_date}&end_date={end_date}"
        f"&hourly={var_str}"
        "&timezone=UTC"
    )

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                data = response.json()
                return data.get("hourly", {})
            else:
                print(f"[{attempt}] ❌ Error {response.status_code}: {response.text}")
        except Exception as e:
            print(f"[{attempt}] ⚠️ Error de conexión: {e}")

        time.sleep(2 ** attempt)  # Espera exponencial entre reintentos

    raise RuntimeError(f"❌ Fallo al descargar datos tras {retries} intentos para {lat}, {lon}")
"""

with open("data_fetcher.py", "w") as f:
    f.write(data_fetcher_code)

print("✅ Archivo data_fetcher.py creado correctamente.")

✅ Archivo data_fetcher.py creado correctamente.


# BLOQUE 3 — Crear archivo processor.py

In [33]:
processor_code = """
# processor.py

import pandas as pd
import numpy as np
import config

def build_hourly_dataframe(hourly_data):
    df = pd.DataFrame({
        "datetime": pd.to_datetime(hourly_data.get("time", [])),
        "T": hourly_data.get("temperature_2m"),
        "RH": hourly_data.get("relativehumidity_2m"),
        "u10": hourly_data.get("windspeed_10m"),
        "Rs_W_m2": hourly_data.get("shortwave_radiation"),
        "precip_mm": hourly_data.get("precipitation"),
    })
    df = df.set_index("datetime").sort_index()
    return df

def aggregate_daily_from_hourly(df_hourly):
    df = df_hourly.copy()
    df["doy"] = df.index.dayofyear
    df["T_mean"] = df["T"]
    df["T_max"] = df["T"]
    df["T_min"] = df["T"]
    df["RH_mean"] = df["RH"]
    df["u2"] = df["u10"] * 0.748  # Corrección por altura (10m -> 2m)
    df["Rs"] = df["Rs_W_m2"] * 0.0864  # W/m² -> MJ/m²/día

    df_daily = df.resample("D").agg({
        "T_mean": "mean",
        "T_max": "max",
        "T_min": "min",
        "RH_mean": "mean",
        "u2": "mean",
        "Rs": "mean",
        "precip_mm": "sum",
        "doy": "first"
    })

    return df_daily

# Curva fenológica para pistacho (puede modificarse)
def kc_pistachio(doy):
    if doy <= 30:
        return 0.4
    elif doy <= 120:
        return 0.4 + (0.9 - 0.4) * (doy - 30) / 90
    elif doy <= 250:
        return 0.9
    elif doy <= 300:
        return 0.9 - (0.9 - 0.7) * (doy - 250) / 50
    elif doy <= 330:
        return 0.7
    else:
        return 0.4

# Fórmula FAO-56 para ETo diario
def eto_fao56(row, lat):
    from math import radians, sin, cos, tan, acos, exp, sqrt, pi

    T = row["T_mean"]
    RH = row["RH_mean"]
    u2 = row["u2"]
    Rs = row["Rs"]
    doy = row["doy"]

    delta = 0.409 * sin(2 * pi * doy / 365 - 1.39)
    phi = radians(lat)
    dr = 1 + 0.033 * cos(2 * pi * doy / 365)
    omega = acos(-tan(phi) * tan(delta))
    Ra = (24 * 60 / pi) * config.GSC * dr * (
        omega * sin(phi) * sin(delta) + cos(phi) * cos(delta) * sin(omega)
    )
    Rso = (0.75 + 2e-5 * 0) * Ra
    es = 0.6108 * exp((17.27 * T) / (T + 237.3))
    ea = es * (RH / 100)
    Rns = (1 - config.ALBEDO) * Rs
    Tk = T + 273.16
    fcd = 1.35 * min(1, Rs / Rso) - 0.35
    Rnl = config.SIGMA * (Tk ** 4) * (0.34 - 0.14 * sqrt(ea)) * fcd
    Rn = Rns - Rnl
    delta_svp = (4098 * es) / ((T + 237.3) ** 2)
    gamma = 0.665e-3 * 101.3
    eto = (0.408 * delta_svp * Rn + gamma * (900 / (T + 273)) * u2 * (es - ea)) / \
          (delta_svp + gamma * (1 + 0.34 * u2))
    return max(0, eto)

def add_agronomic_indicators(df_daily, lat):
    df = df_daily.copy()
    df["ETo"] = df.apply(lambda r: eto_fao56(r, lat), axis=1)
    df["Kc"] = df["doy"].apply(kc_pistachio)
    df["ETc"] = df["ETo"] * df["Kc"]
    df["GDD"] = np.maximum(df["T_mean"] - config.GDD_BASE, 0)
    df["heat_stress"] = (df["T_max"] >= config.HEAT_STRESS_THRESHOLD).astype(int)
    df["frost_event"] = (df["T_min"] <= config.FROST_THRESHOLD).astype(int)
    df["irrigation_need_mm"] = df["ETc"] - df["precip_mm"]
    return df
"""

# Escribimos el archivo
with open("processor.py", "w") as f:
    f.write(processor_code)

print("✅ Archivo processor.py sobrescrito correctamente.")


✅ Archivo processor.py sobrescrito correctamente.


# BLOQUE 4 — Crear archivo locations.py

In [34]:
locations_code = """
# locations.py

# Diccionario de ubicaciones (puedes añadir más si quieres)
LOCATIONS = {
    "Chihuahua_MX": {
        "lat": 27.0398,
        "lon": -105.2063,
        "hemisphere": "north"
    },
    "Saudi Arabia Point II.Sakaka": {
        "lat": 29.7973,
        "lon": 39.9536,
        "hemisphere": "north"
    }
}

# Lista de sitios seleccionados para análisis
SELECTED_SITES = list(LOCATIONS.keys())
"""

# Guardar el archivo
with open("locations.py", "w") as f:
    f.write(locations_code)

print("✅ Archivo locations.py creado correctamente.")


✅ Archivo locations.py creado correctamente.


In [5]:
import shutil
import os
shutil.rmtree("data", ignore_errors=True)
os.makedirs("data", exist_ok=True)

# BLOQUE 5 — load_or_fetch.py

In [35]:
import os
import pandas as pd
import config
import locations
import data_fetcher
import processor

# 🧽 Limpiar archivos de datos NO seleccionados
selected_prefixes = [site.replace(" ", "_") for site in locations.SELECTED_SITES]
data_files = os.listdir("data")

for file in data_files:
    if not any(file.startswith(prefix) for prefix in selected_prefixes):
        os.remove(os.path.join("data", file))
        print(f"🗑️ Archivo eliminado: {file}")

def load_or_fetch(location_name, force_download=False):
    """
    Carga datos climáticos procesados desde disco o los descarga y procesa si no existen.
    """
    # Ruta del archivo
    filename = f"data/{location_name.replace(' ', '_')}.pkl"

    # Si el archivo existe y no se fuerza descarga, lo cargamos
    if os.path.exists(filename) and not force_download:
        print(f"📂 Cargando datos locales: {filename}")
        return pd.read_pickle(filename)

    # Si no existe o se fuerza descarga, procedemos
    print(f"🌐 Descargando datos para: {location_name}")
    info = locations.LOCATIONS[location_name]
    lat, lon = info["lat"], info["lon"]

    # Paso 1: Descargar datos horarios
    hourly = data_fetcher.fetch_hourly_era5(lat, lon, f"{config.START_YEAR}-01-01", f"{config.END_YEAR}-12-31")

    # Paso 2: Construir DataFrame horario
    df_hourly = processor.build_hourly_dataframe(hourly)

    # Paso 3: Agregar a diario y añadir variables agronómicas
    df_daily = processor.aggregate_daily_from_hourly(df_hourly)
    df_agro = processor.add_agronomic_indicators(df_daily, lat)
    df_agro["location"] = location_name

    # Paso 4: Guardar en .pkl y .csv
    os.makedirs("data", exist_ok=True)
    df_agro.to_csv(filename.replace(".pkl", ".csv"))
    df_agro.to_pickle(filename)
    print(f"💾 Datos actualizados y guardados para {location_name}")

    return df_agro


# BLOQUE 6 — Generación de archivos .pkl/.csv para cada ubicación

In [ ]:
import time
import os

# Asegúrate de haber importado antes:
# import locations, config, load_or_fetch (ya hecho en bloques anteriores)

# Iterar por cada localización seleccionada
for site_name in locations.SELECTED_SITES:
    print(f"📥 Procesando: {site_name}")

    # Descargar y procesar datos, forzando descarga si se desea
    df_agro = load_or_fetch(site_name, force_download=True)

    # Crear carpeta 'data' si no existe
    os.makedirs("data", exist_ok=True)

    # Guardar los datos procesados en .csv y .pkl
    csv_path = f"data/{site_name.replace(' ', '_')}.csv"
    pkl_path = f"data/{site_name.replace(' ', '_')}.pkl"
    df_agro.to_csv(csv_path)
    df_agro.to_pickle(pkl_path)

    print(f"💾 Archivos guardados: {csv_path} / {pkl_path}")

    # Esperar 2 segundos entre iteraciones
    time.sleep(60)


📥 Procesando: Chihuahua_MX
🌐 Descargando datos para: Chihuahua_MX


# BLOQUE 7 — Cargar datos procesados desde archivos .pkl y limpiar archivos no usados

In [26]:
import os
import pandas as pd
import locations

# Función para cargar desde .pkl o volver a descargar si se requiere
def load_or_fetch(location_name, force_download=False):
    import config, data_fetcher, processor

    filename = f"data/{location_name.replace(' ', '_')}.pkl"

    if os.path.exists(filename) and not force_download:
        print(f"📂 Cargando datos locales: {filename}")
        return pd.read_pickle(filename)

    print(f"🌐 Descargando datos para: {location_name}")
    info = locations.LOCATIONS[location_name]
    lat, lon = info["lat"], info["lon"]

    hourly = data_fetcher.fetch_hourly_era5(lat, lon, f"{config.START_YEAR}-01-01", f"{config.END_YEAR}-12-31")
    df_hourly = processor.build_hourly_dataframe(hourly)
    df_daily = processor.aggregate_daily_from_hourly(df_hourly)
    df_agro = processor.add_agronomic_indicators(df_daily, lat)
    df_agro["location"] = location_name

    os.makedirs("data", exist_ok=True)
    df_agro.to_csv(filename.replace(".pkl", ".csv"))
    df_agro.to_pickle(filename)
    print(f"💾 Datos actualizados y guardados para {location_name}")

    return df_agro

# 📥 Cargar datos de todas las localizaciones seleccionadas
df_all_sites = pd.concat([
    load_or_fetch(site) for site in locations.SELECTED_SITES
], ignore_index=False)

# 🔒 Filtrar solo las ubicaciones activas (por si hay residuos antiguos en disco)
df_all_sites = df_all_sites[df_all_sites["location"].isin(locations.SELECTED_SITES)]

# 🧽 Limpiar archivos de datos no seleccionados
selected_names = [site.replace(" ", "_") for site in locations.SELECTED_SITES]
for file in os.listdir("data"):
    if not any(name in file for name in selected_names):
        os.remove(os.path.join("data", file))
        print(f"🗑️ Archivo eliminado: {file}")


📂 Cargando datos locales: data/Chihuahua_MX.pkl
📂 Cargando datos locales: data/Saudi_Arabia_Point_II.Sakaka.pkl


In [9]:
print(df_all_sites.columns)



Index(['T_mean', 'T_max', 'T_min', 'RH_mean', 'u2', 'Rs', 'precip_mm', 'doy',
       'ETo', 'Kc', 'ETc', 'GDD', 'heat_stress', 'frost_event',
       'irrigation_need_mm', 'location'],
      dtype='object')


In [10]:
print(df_all_sites.index.to_series().diff().value_counts())


datetime
1 days        18262
-9131 days        1
Name: count, dtype: int64


# Bloque 8. Crear analyzer.py

In [27]:
analyzer_code = """
import pandas as pd

# --- Resumen anual de variables clave ---
def summarize_annual(df):
    df = df.copy()
    df["year"] = df.index.year
    grouped = df.groupby(["year"]).agg({
        "ETo": "sum",
        "ETc": "sum",
        "GDD": "sum",
        "heat_stress": "sum",
        "frost_event": "sum",
        "precip_mm": "sum",
        "irrigation_need_mm": "sum"
    }).reset_index()
    return grouped

# --- Resumen mensual de variables clave ---
def summarize_monthly(df):
    df = df.copy()
    df["year"] = df.index.year
    df["month"] = df.index.month
    grouped = df.groupby(["year", "month"]).agg({
        "ETo": "sum",
        "ETc": "sum",
        "GDD": "sum",
        "heat_stress": "sum",
        "frost_event": "sum",
        "precip_mm": "sum",
        "irrigation_need_mm": "sum"
    }).reset_index()
    return grouped

# --- Resumen de temperatura y precipitación por periodo ---
def summarize_temperature_precip(df, freq="A"):
    df = df.copy()
    df["period"] = df.index.to_period(freq)
    summary = df.groupby(["location", "period"]).agg({
        "T_mean": "mean",
        "T_max": "mean",
        "T_min": "mean",
        "precip_mm": "sum"
    }).reset_index()
    summary["period"] = summary["period"].astype(str)
    return summary

# --- Estimación de horas frío según Weinberger (media de diciembre y enero) ---
def estimate_chill_hours_weinberger(df):
    import numpy as np
    import pandas as pd

    # Tabla original, en orden correcto (mayor T → menos frío)
    WEINBERGER_T = [13.2, 12.3, 11.4, 10.6, 9.8, 8.3, 7.6, 6.9, 6.3]
    WEINBERGER_HF = [450, 550, 650, 750, 850, 950, 1050, 1150, 1350]

    df = df.copy()
    df["year"] = df.index.year
    df["month"] = df.index.month

    df_winter = df[df["month"].isin([12, 1])]
    results = []

    for (location, year), group in df_winter.groupby(["location", "year"]):
        t_dec = group[group["month"] == 12]["T_mean"].mean()
        t_jan = group[group["month"] == 1]["T_mean"].mean()

        t_mean_dj = None
        hf_estimate = None

        if pd.notna(t_dec) and pd.notna(t_jan):
            t_mean_dj = (t_dec + t_jan) / 2
            hf_estimate = float(np.interp(t_mean_dj, WEINBERGER_T, WEINBERGER_HF))

        results.append({
            "location": location,
            "year": year,
            "weinberger_chill": hf_estimate,
            "t_mean_dj": t_mean_dj
        })

    return pd.DataFrame(results))

# --- Cálculo clásico de UC (Grados Día Acumulados > 7°C) ---
def calculate_classic_uc(df):
    df = df.copy()
    df["year"] = df.index.year
    df["UC"] = df["GDD"]
    uc_annual = df.groupby(["location", "year"])["UC"].sum().reset_index()
    return uc_annual
"""

with open("analyzer.py", "w") as f:
    f.write(analyzer_code)

print("✅ analyzer.py guardado correctamente.")


✅ analyzer.py guardado correctamente.


# BLOQUE 9 — Generar resúmenes climáticos anuales y mensuales

In [28]:
# Calculate Weinberger chill hours
df_weinberger_chill = analyzer.estimate_chill_hours_weinberger(df_all_sites)


NameError: name 'analyzer' is not defined

In [29]:
import analyzer
import importlib
import pandas as pd

# 🔁 Recarga del módulo por si se modificó recientemente
importlib.reload(analyzer)

# 📊 Crear resumen ANUAL de temperatura y precipitación por ubicación
df_temp_summary_annual = analyzer.summarize_temperature_precip(df_all_sites, freq="A")

# 📊 Crear resumen MENSUAL de temperatura y precipitación por ubicación
df_temp_summary_monthly = analyzer.summarize_temperature_precip(df_all_sites, freq="M")

# 📅 Añadir columnas auxiliares para análisis mensual
df_all_sites["month"] = df_all_sites.index.month
df_all_sites["adjusted_month"] = df_all_sites.apply(
    lambda row: (row["month"] + 6 - 1) % 12 + 1 if locations.LOCATIONS[row["location"]]["hemisphere"] == "south" else row["month"],
    axis=1
)

# 📊 Cálculo de temperaturas máximas y mínimas mensuales (medias)
monthly_temp = df_all_sites.groupby(["location", "month"]).agg({
    "T_max": "mean",
    "T_min": "mean"
}).reset_index()

# Confirmación de estructuras creadas
print("✅ Resúmenes climáticos generados:")
print(" - df_temp_summary_annual → Temperatura y precipitación anual")
print(" - df_temp_summary_monthly → Temperatura y precipitación mensual")
print(" - monthly_temp → Temperaturas máximas y mínimas mensuales")


SyntaxError: unmatched ')' (analyzer.py, line 82)

# Bloque 10 – Gráficos: Temperaturas y Precipitación

In [30]:
import numpy as np
import matplotlib.pyplot as plt

# 📈 Aseguramos que la columna 'period' esté en formato datetime
df_temp_summary_annual["period"] = pd.to_datetime(df_temp_summary_annual["period"], format="%Y")

# --- 1. Annual Average Temperature ---
plt.figure(figsize=(10, 5))
for loc in df_temp_summary_annual["location"].unique():
    subset = df_temp_summary_annual[df_temp_summary_annual["location"] == loc]
    plt.plot(subset["period"], subset["T_mean"], marker='o', label=loc)
    # Linea de tendencia
    z = np.polyfit(subset["period"].dt.year, subset["T_mean"], 1)
    plt.plot(subset["period"], np.polyval(z, subset["period"].dt.year), linestyle='--')

plt.title("Annual Average Temperature (°C)")
plt.xlabel("Year")
plt.ylabel("°C")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("01_Annual Average Temperature.png")
plt.show()

# --- 2. Annual Total Precipitation ---
plt.figure(figsize=(10, 5))
for loc in df_temp_summary_annual["location"].unique():
    subset = df_temp_summary_annual[df_temp_summary_annual["location"] == loc]
    plt.plot(subset["period"], subset["precip_mm"], marker='s', label=loc)
    z = np.polyfit(subset["period"].dt.year, subset["precip_mm"], 1)
    plt.plot(subset["period"], np.polyval(z, subset["period"].dt.year), linestyle='--')

plt.title("Annual Total Precipitation (mm)")
plt.xlabel("Year")
plt.ylabel("mm")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("02_Annual Total Precipitation.png")
plt.show()

# --- 3. Monthly Average of Max Temperatures ---
plt.figure(figsize=(10, 5))
for loc in monthly_temp["location"].unique():
    subset = monthly_temp[monthly_temp["location"] == loc]
    plt.plot(subset["month"], subset["T_max"], marker='o', label=loc)

plt.title("Monthly Average Maximum Temperature")
plt.xlabel("Month")
plt.ylabel("°C")
plt.xticks(range(1, 13))
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("03_Monthly Average Maximum Temperature.png")
plt.show()

# --- 4. Monthly Average of Min Temperatures ---
plt.figure(figsize=(10, 5))
for loc in monthly_temp["location"].unique():
    subset = monthly_temp[monthly_temp["location"] == loc]
    plt.plot(subset["month"], subset["T_min"], marker='s', label=loc)

plt.title("Monthly Average Minimum Temperature")
plt.xlabel("Month")
plt.ylabel("°C")
plt.xticks(range(1, 13))
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("04_Monthly Average Minimum Temperature.png")
plt.show()


NameError: name 'df_temp_summary_annual' is not defined

# Bloque 11 – Chill Hours & GDD

In [ ]:

import matplotlib.pyplot as plt
import numpy as np

df_weinberger_chill = analyzer.estimate_chill_hours_weinberger(df_all_sites)

# --- 📊 Gráfico: Temperatura media de diciembre+enero (Weinberger) ---
plt.figure(figsize=(12, 6))

for location in df_weinberger_chill["location"].unique():
    subset = df_weinberger_chill[df_weinberger_chill["location"] == location]
    years = subset["year"]
    t_mean_dj = subset["t_mean_dj"]

    # Serie principal
    plt.plot(years, t_mean_dj, marker='o', label=f"{location} (Tmean DJ)")

    # Línea de tendencia
    z = np.polyfit(years, t_mean_dj, 1)
    p = np.poly1d(z)
    plt.plot(years, p(years), linestyle="--", alpha=0.6, label=f"{location} (Trend)")

plt.title("Average Temperature of December + January (Weinberger Input)")
plt.xlabel("Year")
plt.ylabel("Average Temperature [°C]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("05_weinberger_chill.png")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Asegúrate de haber calculado previamente:
# weinberger_results = analyzer.estimate_chill_hours_weinberger(df_all_sites)

plt.figure(figsize=(12, 6))

for location in weinberger_results["location"].unique():
    subset = weinberger_results[weinberger_results["location"] == location]
    years = subset["year"].values
    chills = subset["weinberger_chill"].values

    # Serie principal
    plt.plot(years, chills, marker='o', label=f"{location} (Chill Hours)")

    # Línea de tendencia
    z = np.polyfit(years, chills, 1)
    p = np.poly1d(z)
    plt.plot(years, p(years), linestyle="--", alpha=0.6, label=f"{location} (Trend)")

    # Ecuación al final
    plt.text(
        years[-1], p(years)[-1],
        f"y = {z[0]:.2f}x + {z[1]:.1f}",
        fontsize=9, ha='right', va='bottom', alpha=0.7
    )

plt.title("Estimated Chill Hours. Weinberger Method (2000–2024)")
plt.xlabel("Year")
plt.ylabel("Chill Hours [Weinberger scale]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("06_weinberger_chill_fixed.png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Curva original Weinberger
t_vals = [6.3, 6.9, 7.6, 8.3, 9.8, 10.6, 11.4, 12.3, 13.2]
chill_vals = [1350, 1150, 1050, 950, 850, 750, 650, 550, 450]

plt.figure(figsize=(10, 6))
plt.plot(t_vals, chill_vals, marker="o", label="Curva Weinberger original")
plt.xlabel("Tmedia diciembre+enero [°C]")
plt.ylabel("Horas Frío (escala Weinberger)")
plt.title("Curva Weinberger de estimación de frío")

# Añadir tus datos por ubicación
for location in df_weinberger_chill["location"].unique():
    subset = df_weinberger_chill[df_weinberger_chill["location"] == location]
    plt.scatter(subset["t_mean_dj"], subset["weinberger_chill"], label=f"{location} (datos)", alpha=0.7)

plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def calculate_classic_uc(df):
    df = df.copy()
    df["year"] = df.index.year
    df["month"] = df.index.month

    # Filtrar meses entre abril y septiembre
    df_spring_summer = df[df["month"].between(4, 9)]

    results = []

    for (location, year), group in df_spring_summer.groupby(["location", "year"]):
        mtmm = group.groupby("month")["T_max"].mean().mean()
        mtmn = group.groupby("month")["T_min"].mean().mean()

        if pd.notna(mtmm) and pd.notna(mtmn):
            uc = ((mtmm + mtmn) / 2) * 183
        else:
            uc = None

        results.append({
            "location": location,
            "year": year,
            "UC": uc
        })

    return pd.DataFrame(results)

# --- Calcular UC ---
uc_results = calculate_classic_uc(df_all_sites)

# --- Graficar ---
plt.figure(figsize=(12, 6))
for location in uc_results["location"].unique():
    subset = uc_results[uc_results["location"] == location]
    years = subset["year"]
    uc_values = subset["UC"]

    # Serie principal
    plt.plot(years, uc_values, marker='o', label=f"{location} (UC)")

    # Línea de tendencia
    z = np.polyfit(years, uc_values, 1)
    p = np.poly1d(z)
    plt.plot(years, p(years), linestyle="--", alpha=0.6, label=f"{location} (Trend)")

plt.title("Annual Heat Units (UC) [Apr–Sep]")
plt.xlabel("Year")
plt.ylabel("Heat Units (UC)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("07_classic_uc.png")
plt.show()





In [ ]:
@@import matplotlib.pyplot as plt
import numpy as np
from locations import LOCATIONS

# --- Definir meses críticos para cada umbral y hemisferio ---
CRITICAL_MONTHS = {
    "north": {
        ">35C": [6, 7, 8],
        ">38C": [7, 8],
        "<0C":  [12, 1, 2],
        "<-2C": [1, 2],
    },
    "south": {
        ">35C": [12, 1, 2],
        ">38C": [1, 2],
        "<0C":  [6, 7, 8],
        "<-2C": [6, 7],
    }
}

# --- Crear copia y añadir columnas auxiliares ---
df_ext = df_all_sites.copy()
df_ext["year"] = df_ext.index.year
df_ext["month"] = df_ext.index.month
df_ext["hemisphere"] = df_ext["location"].map(lambda loc: LOCATIONS[loc]["hemisphere"])

# --- Umbrales a analizar ---
extreme_cols = [">35C", ">38C", "<0C", "<-2C"]

# --- Marcar días extremos (en meses críticos)
for col in extreme_cols:
    def is_extreme(row):
        months = CRITICAL_MONTHS[row["hemisphere"]][col]
        if ">" in col:
            threshold = float(col.replace(">", "").replace("C", ""))
            return (row["T_max"] > threshold) and (row["month"] in months)
        else:
            threshold = float(col.replace("<", "").replace("C", ""))
            return (row["T_min"] < threshold) and (row["month"] in months)

    df_ext[col] = df_ext.apply(is_extreme, axis=1).astype(int)

# --- Generar gráficos para cada variable ---
for col in extreme_cols:
    df_plot = df_ext[df_ext[col] == 1].groupby(["location", "year"]).size().reset_index(name="extreme_days")

    plt.figure(figsize=(12, 6))
    for location in df_plot["location"].unique():
        subset = df_plot[df_plot["location"] == location]
        years = subset["year"]
        values = subset["extreme_days"]

        # Serie principal
        plt.plot(years, values, marker='o', label=f"{location} ({col})")

        # Línea de tendencia si hay suficientes puntos
        if len(subset) > 1:
            z = np.polyfit(years, values, 1)
            p = np.poly1d(z)
            plt.plot(years, p(years), linestyle="--", alpha=0.6, label=f"{location} trend")

    plt.title(f"Extreme Days {col} (critical months only)")
    plt.xlabel("Year")
    plt.ylabel("Number of Extreme Days")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"extreme_days_{col.replace('<','lt').replace('>','gt')}.png")
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Preparar datos ---
df = df_all_sites.copy()
df["year"] = df.index.year

# Agrupar por localización y año
annual_summary = df.groupby(["location", "year"])[["ETc", "irrigation_need_mm"]].sum().reset_index()

# --- Función auxiliar para añadir línea de tendencia ---
def add_trend_line(x, y, color, label=None):
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    plt.plot(x, p(x), linestyle='--', alpha=0.6, color=color, label=label)

# --- Gráfico 1: ETc anual ---
plt.figure(figsize=(12, 6))
for i, loc in enumerate(annual_summary["location"].unique()):
    subset = annual_summary[annual_summary["location"] == loc]
    x = subset["year"]
    y = subset["ETc"]
    plt.plot(x, y, marker="o", label=loc)
    add_trend_line(x, y, plt.gca().lines[-1].get_color())

plt.title("Annual Crop Evapotranspiration (ETc)")
plt.xlabel("Year")
plt.ylabel("ETc [mm]")
plt.grid(True)
plt.legend(title="Location")
plt.tight_layout()
plt.savefig("09_annual_etc.png")
plt.show()

# --- Gráfico 2: Necesidad neta de riego anual ---
plt.figure(figsize=(12, 6))
for i, loc in enumerate(annual_summary["location"].unique()):
    subset = annual_summary[annual_summary["location"] == loc]
    x = subset["year"]
    y = subset["irrigation_need_mm"]
    plt.plot(x, y, marker="s", label=loc)
    add_trend_line(x, y, plt.gca().lines[-1].get_color())

plt.title("Annual Irrigation Need (ETc - Precipitation)")
plt.xlabel("Year")
plt.ylabel("Irrigation Need [mm]")
plt.grid(True)
plt.legend(title="Location")
plt.tight_layout()
plt.savefig("10_annual_irrigation_need.png")
plt.show()


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
import matplotlib.image as mpimg
import os
import glob

# --- Parámetros de entrada ---
sites = list(df_all_sites["location"].unique())
site_names_str = ", ".join(sites).replace("_", " ")
pdf_title = f"Climatic Report for {site_names_str} (2000–2024)"
pdf_filename = f"Climatic_Report_{'_'.join(sites)}.pdf"

with PdfPages(pdf_filename) as pdf:
    # Portada
    fig, ax = plt.subplots(figsize=(11.7, 8.3))  # A4 horizontal
    ax.axis("off")
    ax.text(0.5, 0.72, "Climatic Report", fontsize=26, weight="bold", ha="center")
    ax.text(0.5, 0.65, f"for {site_names_str} (2000–2024)", fontsize=18, weight="bold", ha="center")
    today_str = datetime.today().strftime("%Y-%m-%d")
    ax.text(0.5, 0.45, f"Generated on {today_str}", ha="center", fontsize=10)
    ax.text(0.5, 0.40, f"Included locations: {', '.join(sites)}", ha="center", fontsize=10)
    pdf.savefig(fig)
    plt.close()

    # Añadir cada gráfico numerado al PDF
    for i in range(1, 30):  # hasta 30 por si generas más
        for img_path in sorted(glob.glob(f"{i:02d}_*.png")):
            img = mpimg.imread(img_path)
            fig, ax = plt.subplots(figsize=(11.7, 8.3))  # A4 horizontal
            ax.axis("off")
            ax.imshow(img)
            fig.text(0.5, 0.95, pdf_title, fontsize=12, ha="center", weight="bold")
            pdf.savefig(fig)
            plt.close()

print(f"✅ PDF generado: '{pdf_filename}'")
